In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
import sys
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Import the custom loader utilities
from har_dataset_loader import (
    load_dataset,
    get_input_columns,
    get_label_column,
    get_subject_column,
    DATASET_CONFIGS
)

# Add ISW to path for the TinyHAR model import
sys.path.append('./ISWC22-HAR')
try:
    from ISW.models.TinyHAR import TinyHAR_Model
except ImportError:
    print("Error: Could not find TinyHAR_Model in ./ISWC22-HAR. Please check path.")

# ======================= Early Stopping =======================
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta

    def __call__(self, val_loss, model, path):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path)
            self.counter = 0
    
    def save_checkpoint(self, val_loss, model, path):
        os.makedirs(path, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(path, 'best_model.pth'))
        self.val_loss_min = val_loss

# ======================= Configuration =======================
class Args:
    def __init__(self, dataset_name):
        self.dataset_name = dataset_name
        self.window_length = 300 
        self.filter_num = 20
        self.nb_conv_layers = 4
        self.filter_size = 5
        self.dropout = 0.1
        self.cross_channel_interaction_type = "transformer"
        self.cross_channel_aggregation_type = "FC"
        self.temporal_info_interaction_type = "lstm"
        self.temporal_info_aggregation_type = "FC"
        self.batch_size = 64
        self.learning_rate = 0.001
        self.train_epochs = 100
        self.early_stop_patience = 15
        self.use_gpu = torch.cuda.is_available()
        self.gpu = 0
        self.seed = 42
        self.output_dir = f'./results/tinyhar_{dataset_name}_results'

# ======================= Dataset Wrapper =======================
class HAR_Torch_Dataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        # Shape: (1, Window, Channels)
        window = np.expand_dims(self.x[idx], axis=0) 
        return torch.tensor(window, dtype=torch.float64), torch.tensor(self.y[idx], dtype=torch.long)

# ======================= Data Preparation =======================
def prepare_data_for_loso(args):
    """Loads dataset, filters common labels across subjects, and normalizes."""
    df_cleaned = load_dataset(args.dataset_name, window_length=args.window_length)
    
    subject_col = get_subject_column(args.dataset_name)
    label_col = get_label_column(args.dataset_name)
    signal_cols = get_input_columns(args.dataset_name)
    
    # Common labels across all subjects logic
    per_subject_labels = df_cleaned.groupby(subject_col)[label_col].apply(set)
    common_labels = set.intersection(*per_subject_labels)
    df_cleaned = df_cleaned[df_cleaned[label_col].isin(common_labels)].reset_index(drop=True)
    
    # Map labels to 0-N
    label_mapping = {label: idx for idx, label in enumerate(sorted(list(common_labels)))}
    df_cleaned[label_col] = df_cleaned[label_col].map(label_mapping)
    
    subjects = df_cleaned[subject_col].unique()
    subject_data = {}
    
    for sub in subjects:
        sub_df = df_cleaned[df_cleaned[subject_col] == sub]
        data_block = [np.stack(sub_df[col].values) for col in signal_cols]
        features = np.stack(data_block, axis=-1) 
        
        # Subject-wise Normalization
        mean, std = features.mean(axis=(0, 1)), features.std(axis=(0, 1))
        features = (features - mean) / (std + 1e-8)
        
        subject_data[sub] = {'features': features, 'labels': sub_df[label_col].values}
        
    return subject_data, subjects, len(common_labels), signal_cols

# ======================= Evaluation =======================
def evaluate(model, loader, criterion, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
            
    return {
        'loss': total_loss / len(loader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision_weighted': precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        'precision_macro': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall_weighted': recall_score(all_labels, all_preds, average='weighted', zero_division=0),
        'recall_macro': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'predictions': all_preds,
        'labels': all_labels
    }

# ======================= Training Fold =======================
def train_loso_fold(train_subjects, test_subject, subject_data, args, device, fold_num, num_classes, num_channels):
    print(f"\n{'='*60}\nLOSO Fold {fold_num}: Testing on Subject {test_subject}\n{'='*60}")
    
    train_x = np.vstack([subject_data[s]['features'] for s in train_subjects])
    train_y = np.concatenate([subject_data[s]['labels'] for s in train_subjects])
    test_x = subject_data[test_subject]['features']
    test_y = subject_data[test_subject]['labels']
    
    train_loader = DataLoader(HAR_Torch_Dataset(train_x, train_y), batch_size=args.batch_size, shuffle=True)
    test_loader = DataLoader(HAR_Torch_Dataset(test_x, test_y), batch_size=args.batch_size)

    # Initialize TinyHAR
    model = TinyHAR_Model(
        input_shape=(1, 1, args.window_length, num_channels),
        number_class=num_classes,
        filter_num=args.filter_num,
        nb_conv_layers=args.nb_conv_layers,
        filter_size=args.filter_size,
        cross_channel_interaction_type=args.cross_channel_interaction_type,
        cross_channel_aggregation_type=args.cross_channel_aggregation_type,
        temporal_info_interaction_type=args.temporal_info_interaction_type,
        temporal_info_aggregation_type=args.temporal_info_aggregation_type,
        dropout=args.dropout
    ).double().to(device)

    num_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)
    criterion = nn.CrossEntropyLoss()
    fold_path = os.path.join(args.output_dir, f'fold_{fold_num}')
    stopper = EarlyStopping(patience=args.early_stop_patience, verbose=True)

    for epoch in range(args.train_epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
        
        val_res = evaluate(model, test_loader, criterion, device)
        stopper(val_res['loss'], model, fold_path)
        if stopper.early_stop: 
            print("Early stopping triggered!")
            break

    # Load best weights
    model.load_state_dict(torch.load(os.path.join(fold_path, 'best_model.pth')))
    final_results = evaluate(model, test_loader, criterion, device)
    
    print(f"Fold {fold_num} Results - Acc: {final_results['accuracy']:.4f}, F1-Macro: {final_results['f1_macro']:.4f}")
    
    return {
        'test_subject': test_subject,
        'accuracy': final_results['accuracy'],
        'precision_weighted': final_results['precision_weighted'],
        'precision_macro': final_results['precision_macro'],
        'recall_weighted': final_results['recall_weighted'],
        'recall_macro': final_results['recall_macro'],
        'f1_weighted': final_results['f1_weighted'],
        'f1_macro': final_results['f1_macro'],
        'test_loss': final_results['loss'],
        'num_params': num_params
    }

# ======================= Main Loop =======================
def main():
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    all_dataset_names = sorted(list(DATASET_CONFIGS.keys()))
    
    os.makedirs('./results', exist_ok=True)
    global_comparison_list = []

    for name in all_dataset_names:
        print(f"\n{'#'*80}\nRUNNING EXPERIMENT: {name.upper()}\n{'#'*80}")
        args = Args(dataset_name=name)
        
        try:
            subject_data, subjects, num_classes, signal_cols = prepare_data_for_loso(args)
            all_fold_results = []
            
            for fold_num, test_sub in enumerate(subjects, 1):
                train_subs = [s for s in subjects if s != test_sub]
                res = train_loso_fold(train_subs, test_sub, subject_data, args, device, 
                                      fold_num, num_classes, len(signal_cols))
                all_fold_results.append(res)

            # Process summary for this dataset
            results_df = pd.DataFrame(all_fold_results)
            avg_f1 = results_df['f1_macro'].mean()
            avg_acc = results_df['accuracy'].mean()
            
            print(f"\n--- {name.upper()} SUMMARY ---\n{results_df.to_string(index=False)}")
            results_df.to_csv(os.path.join(args.output_dir, f'{name}_loso_summary.csv'), index=False)
            
            global_comparison_list.append({
                'dataset': name,
                'avg_f1_macro': avg_f1,
                'avg_accuracy': avg_acc,
                'subjects': len(subjects),
                'classes': num_classes,
                'channels': len(signal_cols)
            })

        except Exception as e:
            print(f"FAILED DATASET {name}: {e}")

    # Final Overall Summary
    print("\n\n" + "="*80 + "\nPERFORMANCE RANKING (F1-MACRO)\n" + "="*80)
    summary_df = pd.DataFrame(global_comparison_list)
    if not summary_df.empty:
        print(summary_df.sort_values(by='avg_f1_macro', ascending=False).to_string(index=False))
        summary_df.to_csv('./results/tinyhar_dataset_comparison.csv', index=False)

if __name__ == "__main__":
    main()


################################################################################
RUNNING EXPERIMENT: CAPTURE24
################################################################################
[capture24] Loading /media/user/DATA21/Time_Series_Foundation_Models/HAR/HAR_New/test_set/capture24_30hz_full_test.csv ...
[capture24] Raw shape: (9225000, 5)
[capture24] Windows: 30750 (window_length=300)
[capture24] df_cleaned shape: (30750, 6)  (subjects=5, classes=4)

LOSO Fold 1: Testing on Subject P024
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
EarlyStopping counter: 7 out of 15
EarlyStopping counter: 8 out of 15
EarlyStopping counter: 9 out of 15
EarlyStopping counter: 10 out of 15
EarlyStopping counter: 11 out of 15
EarlyStopping counter: 12 out of 15
EarlyStopping counter: 13 out of 15
EarlyStopping counter: 14 out of 15
Ea